In [ ]:
# Install only what the suite imports. Nothing here depends on torch, and the image's own
# transformers and tokenizers are used as they are (5.16.1 / 0.23.1 ran cleanly).
# No -q and a hard timeout: a stalled install must fail loudly instead of hanging the kernel silently.
!timeout 900 pip install --no-deps --retries 2 --timeout 60 "diffusers==0.29.0" "conformer==0.3.2" "s3tokenizer" pyloudnorm omegaconf resemble-perth torchao


In [ ]:
# The dataset mount path and layout are not fixed, so locate the suite rather than assume it.
# An archive upload is unpacked into the working directory first.
import glob, json, os, subprocess, zipfile
hits = glob.glob('/kaggle/input/**/nano_fast/kaggle/suite.py', recursive=True)
if not hits:
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/kaggle/working/src')
    hits = glob.glob('/kaggle/working/src/**/nano_fast/kaggle/suite.py', recursive=True)
suite = hits[0]
root = suite.rsplit('/nano_fast/kaggle/suite.py', 1)[0]
os.makedirs('/kaggle/working/results', exist_ok=True)
# One process per experiment: a compile failure or a poisoned CUDA context stays inside its process.
# Experiments that need reference tokens or audio run the baseline first, in the same process.
# probe checks the decode loop and the verify path for non-finite values; probe_nored repeats it with
# fp16 reduced-precision matmul reduction off. The *_dbg jobs rerun the two device-assert jobs with
# CUDA_LAUNCH_BLOCKING=1 and print every decode position. sampling_eager runs the timed host loop with
# the decode step uncompiled; verify_events times verify with CUDA events instead of the profiler.
jobs = [
    ('env', 'env'),
    ('probe', 'probe'),
    ('probe_nored', 'probe', {'CHATNANO_FP16_REDUCED': '0'}),
    ('baseline', 'baseline'),
    ('attention', 'baseline,attention'),
    ('breakdown', 'breakdown'),
    ('cache', 'cache'),
    ('verify', 'verify'),
    ('verify_dbg', 'verify', {'CUDA_LAUNCH_BLOCKING': '1', 'CHATNANO_TRACE_POS': '1'}),
    ('verify_events', 'verify', {'CHATNANO_VERIFY_EVENTS': '1', 'CUDA_LAUNCH_BLOCKING': '1', 'CHATNANO_TRACE_POS': '1'}),
    ('batch', 'batch'),
    ('sampling', 'sampling'),
    ('sampling_dbg', 'sampling', {'CUDA_LAUNCH_BLOCKING': '1', 'CHATNANO_TRACE_POS': '1'}),
    ('sampling_eager', 'sampling', {'CHATNANO_EAGER_STEP': '1', 'CUDA_LAUNCH_BLOCKING': '1', 'CHATNANO_TRACE_POS': '1'}),
    ('modes', 'modes'),
    ('speculative', 'baseline,speculative'),
    ('vocoder', 'vocoder'),
    ('quant_linear', 'baseline,quant_linear'),
    ('quant_int8', 'baseline,quant_int8'),
    ('quant_int4', 'baseline,quant_int4'),
]
for job in jobs:
    name, only = job[0], job[1]
    extra = job[2] if len(job) > 2 else {}
    env = {**os.environ, 'CHATNANO_SRC': root, 'RESULTS': f'/kaggle/working/results/{name}.json', **extra}
    r = subprocess.run(['python', suite, '--only', only], env=env)
    print('JOB', name, 'exit', r.returncode, flush=True)


In [ ]:
# One file for the whole run; each process wrote its own results/<job>.json.
merged = {}
for f in sorted(glob.glob('/kaggle/working/results/*.json')):
    merged[os.path.basename(f)[:-5]] = json.load(open(f))['experiments']
json.dump(merged, open('/kaggle/working/results_all.json', 'w'), indent=1)
print('merged', sorted(merged))
